# Set PYTHONPATH

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "src"))

# PreProcessing

In [2]:
import re
import matplotlib
matplotlib.use('Agg')

from joblib import Parallel, delayed

from utils import *
from courts.basket import BasketCourt
from processors.voronoi_area import VoronoiPreProcessor
from visualizers.voronoi import VoronoiAreaVisualizer
from visualizers.trajectory import TrajectoryVisualizer


def _resolve_save_dir(base_name: str, analyzer_dir: str, player_type: str) -> str:
    """Resolve the save directory based on filename pattern and player type."""
    if player_type == "tokoha":
        match = re.match(r"basket_S(\d+)T\d+", base_name)
        if match:
            n = int(match.group(1))
            if n >= 4:
                return os.path.join(analyzer_dir, "processed_data/after")
            else:
                return os.path.join(analyzer_dir, "processed_data/before")
    elif player_type == "kokutai":
        match = re.match(r"basket_([A-Z]{2})-S\d+T\d+", base_name)
        if match:
            group = match.group(1)
            return os.path.join(analyzer_dir, f"./processed_data/{group}")
    elif player_type == "yamaha":
        match = re.match(r"basket_G(\d+)-S\d+T\d+", base_name)
        if match:
            n = int(match.group(1))
            return os.path.join(analyzer_dir, f"./processed_data/G{n}")
    raise ValueError(f"Could not resolve save directory for {base_name} with player_type={player_type}")


def process_single_file(file_path: str, analyzer_dir: str, player_type: str) -> None:
    """Process one CSV file: trajectory PNG, Voronoi animation MP4, area pickle."""
    court = BasketCourt()
    base_name = os.path.basename(file_path).replace('.csv', '')
    save_dir_name = _resolve_save_dir(base_name, analyzer_dir, player_type)
    save_dir_name = os.path.join(save_dir_name, base_name)
    file_name = "basketball_court"

    data = get_data_from_csv(file_path)

    # Compute Voronoi once for all frames
    processor = VoronoiPreProcessor(court)
    precomputed = processor.compute_all_frames(data)

    # Trajectory PNG
    trajectory_visualizer = TrajectoryVisualizer(court)
    fig, ax = trajectory_visualizer.draw_trajectories(data, base_name)
    save_plot(fig, save_dir_name, file_name)

    # Voronoi animation (reuse precomputed results)
    voronoi_visualizer = VoronoiAreaVisualizer(court)
    anim = voronoi_visualizer.draw_voronoi_animation(data, interval=100, title=base_name, precomputed_results=precomputed)
    save_voronoi_animation(anim, save_dir_name, file_name)

    # Pickle (reuse precomputed results)
    players_area_trajectories = processor.get_players_area_trajectories(data, precomputed_results=precomputed)
    save_pickle(players_area_trajectories, save_dir_name, "players_area_trajectories")

    plt.close('all')
    print(f"Done: {base_name}")


def recording(data_dir: str, analyzer_dir: str, player_type: str) -> None:
    """
    Record the Voronoi area visualization for a specific player type.

    :param data_dir: Directory containing CSV files.
    :param analyzer_dir: Directory to save analysis results.
    :param player_type: Type of player to visualize.
    """
    file_paths = get_csv_file_path(data_dir)

    Parallel(n_jobs=8, backend='loky', verbose=10)(
        delayed(process_single_file)(fp, analyzer_dir, player_type)
        for fp in file_paths
    )

if __name__ == "__main__":
    DATA_DIRS = ["../../_dataset", "../../_dataset_tokoha", "../../_dataset_yamaha"]
    ANALYZER_DIR = "../analyzers"
    for DIR_NAME in DATA_DIRS:
        if DIR_NAME == "../../_dataset":
            print("Processing kokutai dataset")
            recording(DIR_NAME, ANALYZER_DIR, player_type="kokutai")
        elif DIR_NAME == "../../_dataset_tokoha":
            print("Processing tokoha dataset")
            recording(DIR_NAME, ANALYZER_DIR, player_type="tokoha")
        elif DIR_NAME == "../../_dataset_yamaha":
            print("Processing yamaha dataset")
            recording(DIR_NAME, ANALYZER_DIR, player_type="yamaha")

Processing kokutai dataset
Processing tokoha dataset
Processing yamaha dataset


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   31.9s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:  1.3min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  1.9min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.5min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  3.5min
[Parallel(n_jobs=8)]: Done  45 tasks      | elapsed:  4.3min
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.1min
[Parallel(n_jobs=8)]: Done  69 tasks      | elapsed:  5.9min
[Parallel(n_jobs=8)]: Done  78 out of  84 | elapsed:  6.8min remaining:   31.2s
[Parallel(n_jobs=8)]: Done  84 out of  84 | elapsed:  7.8min finished
